In [ ]:
r0 = float(r_clipped[0])

cir_curve = CIRCurve(
    short_rate=r0,
    theta=theta_hat,
    kappa=kappa_hat,
    beta=beta_hat
)
# was after "residuals from AR1

In [ ]:
# Tenors/maturities in years
tenors = np.array([0.25] + list(range(1, 11)))

# pick a few time indices from term_structure
idx_first  = term_structure.index[0]
idx_middle = term_structure.index[len(term_structure)//2]
idx_last   = term_structure.index[-1]
date_indices = [idx_first, idx_middle, idx_last]

fig, ax = plt.subplots(figsize=(8,5))

for idx in date_indices:
    # observed bootstrapped yields for that date
    y_obs = term_structure.loc[idx, ["y_0_25"] + [f"y_{n}Y" for n in range(1,11)]].values

    # current short rate at that date (3M)
    r_t = float(term_structure.loc[idx, "y_0_25"])

    # build CIR curve at that time
    cir_curve_t = CIRCurve(
        short_rate=r_t,
        theta=theta_hat,
        kappa=kappa_hat,
        beta=beta_hat
    )

    # model-implied yields for all tenors
    y_model = cir_curve_t.zero_rate_vector(tenors)

    # plot
    plt.plot(tenors, y_obs,   marker="o", linestyle="-",  label=f"Obs, t={idx}")
    plt.plot(tenors, y_model, marker="x", linestyle="--", label=f"CIR, t={idx}")

plt.xlabel("Maturity (years)")
plt.ylabel("Yield (cont. comp.)")
plt.title("Bootstrapped vs CIR Model Yields")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

# One or 2 factor sanity check not usefull. comes after eps_hat

In [ ]:
# Comes after first two dataclasses.

rng = np.random.default_rng()

def simulate_chen_scott_2f(params: ChenScott2FParams,
                           y0_1: float,
                           y0_2: float,
                           n_steps: int,
                           dt: float = DT):
    """
    Simulate 2-factor Chen–Scott model.
    Returns factor1, factor2 and short-rate r_t = y1 + y2.
    """
    y1 = np.zeros(n_steps + 1)
    y2 = np.zeros(n_steps + 1)
    r  = np.zeros(n_steps + 1)

    y1[0] = max(y0_1, 1e-8)
    y2[0] = max(y0_2, 1e-8)
    r[0]  = y1[0] + y2[0]

    rho = np.clip(params.rho, -0.999, 0.999)

    for t in range(n_steps):
        # correlated normals
        z1, z2 = rng.normal(), rng.normal()
        e1 = z1
        e2 = rho * z1 + np.sqrt(1 - rho**2) * z2

        # factor 1
        f1 = params.factor1
        y1_pos = max(y1[t], 1e-8)
        drift1 = f1.kappa * (f1.theta - y1_pos) * dt
        diff1  = f1.sigma * np.sqrt(y1_pos) * np.sqrt(dt) * e1
        y1[t+1] = max(y1_pos + drift1 + diff1, 1e-8)

        # factor 2
        f2 = params.factor2
        y2_pos = max(y2[t], 1e-8)
        drift2 = f2.kappa * (f2.theta - y2_pos) * dt
        diff2  = f2.sigma * np.sqrt(y2_pos) * np.sqrt(dt) * e2
        y2[t+1] = max(y2_pos + drift2 + diff2, 1e-8)

        r[t+1] = y1[t+1] + y2[t+1]

    return y1, y2, r